# COMP 469 Lab 01: Vacuum World Agents - Completed Teaching Version

**Week 1** | **AIMA Chapters 1-2** | **3-hour lab instructor code**

This completed version fills in the environment, reflex agent, model-based agent, goal-based BFS agent, experiments, and plotting code. Use it for live teaching, walkthroughs, and instructor preparation.


## 0. Python `.env` Setup

Before opening or running the notebook, activate your course Python environment.

### First-Time Setup

Run these commands from the folder where this notebook is located.

**macOS/Linux:**

```bash
python3 -m venv .env
source .env/bin/activate
python -m pip install --upgrade pip
python -m pip install ipykernel notebook jupyter matplotlib
python -m ipykernel install --user --name comp469 --display-name "COMP 469 (.env)"
```

**Windows PowerShell:**

```powershell
py -m venv .env
.\.env\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install ipykernel notebook jupyter matplotlib
python -m ipykernel install --user --name comp469 --display-name "COMP 469 (.env)"
```

If PowerShell blocks activation, run this once in the same terminal:

```powershell
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process
.\.env\Scripts\Activate.ps1
```

### Returning Later

Always activate the environment before launching Jupyter.

**macOS/Linux:**

```bash
source .env/bin/activate
jupyter notebook
```

**Windows PowerShell:**

```powershell
.\.env\Scripts\Activate.ps1
jupyter notebook
```

In Jupyter, choose the kernel named **COMP 469 (.env)** if it appears.

## 1. Setup Cell

Run this cell first. Do not edit it unless your instructor asks you to.

In [ ]:
import random
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt

Position = Tuple[int, int]
ACTIONS = ["Suck", "Up", "Down", "Left", "Right", "NoOp"]

random.seed(469)
print("Setup complete. You are ready for Lab 01.")

## 2. PEAS Analysis - Teaching Notes

A good answer identifies the performance measure as score maximization: cleaning earns points, actions cost points, and bumps cost extra. The environment is a finite grid with clean/dirty squares and possible obstacles. Actuators are `Suck`, movement actions, and `NoOp`; sensors provide current location and current square status.

This version is partially observable, single-agent, sequential, discrete, and mostly known. It is deterministic after world initialization, though the initial world generation is random.


## 3. Build the Vacuum World Environment - Completed Code

The environment owns the true world state. The agent receives only a percept and returns an action.

This completed environment supports:

- A rectangular grid.
- Clean and dirty squares.
- Optional obstacles.
- A score that rewards cleaning and penalizes action costs.
- A percept containing only the agent's current location and current square status.


In [ ]:
@dataclass
class Percept:
    location: Position
    status: str


class VacuumEnvironment:
    def __init__(
        self,
        width: int = 5,
        height: int = 5,
        dirt_probability: float = 0.30,
        obstacle_probability: float = 0.0,
        start: Position = (0, 0),
        seed: Optional[int] = None,
    ):
        if seed is not None:
            random.seed(seed)

        self.width = width
        self.height = height
        self.start = start
        self.agent_location = start
        self.status: Dict[Position, str] = {}
        self.obstacles = set()
        self.score = 0
        self.steps = 0
        self.cleaned_count = 0

        for x in range(width):
            for y in range(height):
                location = (x, y)
                if location != start and random.random() < obstacle_probability:
                    self.obstacles.add(location)

        for x in range(width):
            for y in range(height):
                location = (x, y)
                if location in self.obstacles:
                    self.status[location] = "Obstacle"
                else:
                    self.status[location] = "Dirty" if random.random() < dirt_probability else "Clean"

        self.status[start] = "Clean"

    def in_bounds(self, location: Position) -> bool:
        x, y = location
        return 0 <= x < self.width and 0 <= y < self.height

    def is_accessible(self, location: Position) -> bool:
        return self.in_bounds(location) and location not in self.obstacles

    def get_percept(self) -> Percept:
        return Percept(self.agent_location, self.status[self.agent_location])

    def get_neighbors(self, location: Position) -> Dict[str, Position]:
        x, y = location
        candidates = {
            "Up": (x, y - 1),
            "Down": (x, y + 1),
            "Left": (x - 1, y),
            "Right": (x + 1, y),
        }
        return {
            action: next_location
            for action, next_location in candidates.items()
            if self.is_accessible(next_location)
        }

    def execute_action(self, action: str) -> None:
        if action not in ACTIONS:
            raise ValueError(f"Unknown action: {action}")

        self.steps += 1
        self.score -= 1

        if action == "NoOp":
            return

        if action == "Suck":
            if self.status[self.agent_location] == "Dirty":
                self.status[self.agent_location] = "Clean"
                self.score += 10
                self.cleaned_count += 1
            return

        x, y = self.agent_location
        moves = {
            "Up": (x, y - 1),
            "Down": (x, y + 1),
            "Left": (x - 1, y),
            "Right": (x + 1, y),
        }
        next_location = moves[action]

        if self.is_accessible(next_location):
            self.agent_location = next_location
        else:
            self.score -= 2

    def count_dirty_squares(self) -> int:
        return sum(1 for value in self.status.values() if value == "Dirty")

    def count_clean_squares(self) -> int:
        return sum(
            1
            for location, value in self.status.items()
            if value == "Clean" and location not in self.obstacles
        )

    def is_done(self) -> bool:
        return self.count_dirty_squares() == 0

    def copy_world(self):
        return dict(self.status), set(self.obstacles)

    def load_world(self, status: Dict[Position, str], obstacles: set) -> None:
        self.status = dict(status)
        self.obstacles = set(obstacles)
        self.agent_location = self.start
        self.score = 0
        self.steps = 0
        self.cleaned_count = 0


env = VacuumEnvironment(width=5, height=5, dirt_probability=0.35, obstacle_probability=0.10, seed=469)
print("Start:", env.agent_location)
print("Percept:", env.get_percept())
print("Dirty squares:", env.count_dirty_squares())
print("Obstacles:", len(env.obstacles))

## 4. Visualize and Inspect the World

Complete the visualization function. The goal is to make debugging easier.

Suggested display choices:

- `A` or blue marker for the agent.
- Dark cells for dirty squares.
- Light cells for clean squares.
- Black cells for obstacles.

In [ ]:
def display_text(env: VacuumEnvironment) -> None:
    for y in range(env.height):
        row = []
        for x in range(env.width):
            location = (x, y)
            if location == env.agent_location:
                row.append("A")
            elif location in env.obstacles:
                row.append("#")
            elif env.status[location] == "Dirty":
                row.append("D")
            else:
                row.append(".")
        print(" ".join(row))
    print(f"Score: {env.score}, Steps: {env.steps}, Dirty left: {env.count_dirty_squares()}")


def plot_environment(env: VacuumEnvironment, title: str = "Vacuum World") -> None:
    grid = []
    for y in range(env.height):
        row = []
        for x in range(env.width):
            location = (x, y)
            if location in env.obstacles:
                row.append(2)
            elif env.status[location] == "Dirty":
                row.append(1)
            else:
                row.append(0)
        grid.append(row)

    plt.figure(figsize=(5, 5))
    plt.imshow(grid, cmap="Greys", vmin=0, vmax=2)
    ax = plt.gca()
    ax.set_xticks(range(env.width))
    ax.set_yticks(range(env.height))
    ax.set_xticks([i - 0.5 for i in range(1, env.width)], minor=True)
    ax.set_yticks([i - 0.5 for i in range(1, env.height)], minor=True)
    ax.grid(which="minor", color="black", linewidth=1)
    ax.scatter([env.agent_location[0]], [env.agent_location[1]], c="tab:blue", s=250)
    ax.set_title(title)
    plt.show()


display_text(env)
plot_environment(env, "Initial Vacuum World")

## 5. Agent Interface and Random Baseline

Every agent will implement `choose_action(percept, environment)`. The random agent is a baseline: it cleans when it sees dirt and otherwise moves randomly.

In [ ]:
class Agent:
    name = "Base Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        raise NotImplementedError


class RandomVacuumAgent(Agent):
    name = "Random Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        if percept.status == "Dirty":
            return "Suck"
        return random.choice(["Up", "Down", "Left", "Right"])


random_agent = RandomVacuumAgent()
print(random_agent.name, "chooses", random_agent.choose_action(env.get_percept(), env))

## 6. Simulation Runner

Complete the runner so that it repeatedly:

1. Gets the current percept.
2. Asks the agent for an action.
3. Applies the action to the environment.
4. Records score and dirty-square history.
5. Stops if the environment is clean or the agent returns `NoOp`.

In [ ]:
def run_simulation(
    agent: Agent,
    env: VacuumEnvironment,
    max_steps: int = 100,
    verbose: bool = False,
) -> dict:
    score_history = []
    dirty_history = []
    action_history = []

    for step in range(max_steps):
        # Get percept, choose action, execute action, and record history.
        percept = env.get_percept()
        action = agent.choose_action(percept, env)

        if action == "NoOp":
            break

        env.execute_action(action)

        score_history.append(env.score)
        dirty_history.append(env.count_dirty_squares())
        action_history.append(action)

        if verbose:
            print(f"step={step:02d} loc={percept.location} status={percept.status} action={action} score={env.score}")

        if env.is_done():
            break

    return {
        "agent": agent.name,
        "score": env.score,
        "steps": env.steps,
        "cleaned": env.cleaned_count,
        "dirty_left": env.count_dirty_squares(),
        "score_history": score_history,
        "dirty_history": dirty_history,
        "action_history": action_history,
    }


test_env = VacuumEnvironment(width=5, height=5, dirt_probability=0.35, obstacle_probability=0.0, seed=101)
test_result = run_simulation(RandomVacuumAgent(), test_env, max_steps=25)
test_result

## 7. Program a Reflex Agent

A reflex agent reacts only to the current percept. It does not remember where it has been.

Your task:

- Return `Suck` if the current square is dirty.
- Otherwise, use a deterministic movement rule.
- Your rule should work reasonably well on an open 5x5 grid.

Hint: A serpentine pattern is a good starting point.

In [ ]:
class ReflexVacuumAgent(Agent):
    name = "Reflex Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        x, y = percept.location

        if percept.status == "Dirty":
            return "Suck"

        # Serpentine sweep: move right on even rows, left on odd rows, down at row ends.
        if y % 2 == 0:
            preferred = "Right" if x < environment.width - 1 else "Down"
        else:
            preferred = "Left" if x > 0 else "Down"

        legal_actions = environment.get_neighbors(percept.location)
        if preferred in legal_actions:
            return preferred

        # Obstacles can break the sweep, so use a deterministic fallback.
        for action in ["Right", "Down", "Left", "Up"]:
            if action in legal_actions:
                return action

        return "NoOp"


reflex_env = VacuumEnvironment(width=5, height=5, dirt_probability=0.35, obstacle_probability=0.0, seed=202)
reflex_result = run_simulation(ReflexVacuumAgent(), reflex_env, max_steps=100)
reflex_result

### Checkpoint 2 - Teaching Notes

The reflex agent uses only the current percept: location and square status. It ignores where it has already been, the full dirt map, and whether a previous movement was useful. The serpentine rule works well on an open rectangular grid, but obstacles can break the pattern and cause inefficient detours. This is a useful place to emphasize the limits of reflex behavior in partially observable sequential environments.


## 8. Program a Model-Based Agent

A model-based agent keeps internal state. In this lab, it should remember visited squares and use that memory to avoid wasting moves.

Your task:

- Store visited locations.
- Clean dirty squares.
- Prefer unvisited accessible neighbors.
- Add a fallback behavior when all neighbors have been visited.

Optional stronger version: keep a stack of previous locations and backtrack when stuck.

In [ ]:
class ModelBasedVacuumAgent(Agent):
    name = "Model-Based Vacuum Agent"

    def __init__(self):
        self.visited = set()
        self.known_status: Dict[Position, str] = {}
        self.path_stack: List[Position] = []

    def action_toward(self, current: Position, target: Position) -> str:
        cx, cy = current
        tx, ty = target
        if tx == cx + 1 and ty == cy:
            return "Right"
        if tx == cx - 1 and ty == cy:
            return "Left"
        if tx == cx and ty == cy + 1:
            return "Down"
        if tx == cx and ty == cy - 1:
            return "Up"
        return "NoOp"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        location = percept.location
        self.visited.add(location)
        self.known_status[location] = percept.status

        if percept.status == "Dirty":
            self.known_status[location] = "Clean"
            return "Suck"

        neighbors = environment.get_neighbors(location)
        unvisited = [
            (action, next_location)
            for action, next_location in neighbors.items()
            if next_location not in self.visited
        ]

        if unvisited:
            action, next_location = unvisited[0]
            self.path_stack.append(location)
            return action

        while self.path_stack:
            previous_location = self.path_stack.pop()
            if previous_location in neighbors.values():
                return self.action_toward(location, previous_location)

        return "NoOp"


model_env = VacuumEnvironment(width=5, height=5, dirt_probability=0.35, obstacle_probability=0.10, seed=303)
model_result = run_simulation(ModelBasedVacuumAgent(), model_env, max_steps=100)
model_result

### Checkpoint 3 - Teaching Notes

The model-based agent stores visited locations, known square statuses, and a backtracking stack. That memory helps it avoid immediately repeating unproductive moves and gives it a systematic exploration pattern. It still does not know the full environment state because its percept only reports the current square. The stack fallback is better than random movement, but it still does not compute globally shortest routes to remaining dirt.


## 9. Program a Goal-Based Agent with BFS

A goal-based agent chooses actions that move it toward a goal. Here, the goal is to reach and clean dirty squares.

For this lab, the goal-based agent is allowed to inspect the full environment object. This makes it less realistic than the model-based agent, but it helps connect agents to search algorithms before Lab 02.

Your task:

- If the current square is dirty, return `Suck`.
- Find all dirty locations.
- Use breadth-first search to find a path to the nearest reachable dirty square.
- Return the first action in that path.

In [ ]:
class GoalBasedVacuumAgent(Agent):
    name = "Goal-Based Vacuum Agent"

    def choose_action(self, percept: Percept, environment: VacuumEnvironment) -> str:
        if percept.status == "Dirty":
            return "Suck"

        dirty_locations = [
            location
            for location, status in environment.status.items()
            if status == "Dirty"
        ]

        if not dirty_locations:
            return "NoOp"

        path = self.bfs_to_nearest_dirty(percept.location, dirty_locations, environment)

        if path:
            return path[0]

        return "NoOp"

    def bfs_to_nearest_dirty(
        self,
        start: Position,
        dirty_locations: List[Position],
        environment: VacuumEnvironment,
    ) -> List[str]:
        dirty_set = set(dirty_locations)
        frontier = deque([(start, [])])
        visited = {start}

        while frontier:
            location, path_so_far = frontier.popleft()

            if location in dirty_set:
                return path_so_far

            for action, next_location in environment.get_neighbors(location).items():
                if next_location not in visited:
                    visited.add(next_location)
                    frontier.append((next_location, path_so_far + [action]))

        return []


goal_env = VacuumEnvironment(width=5, height=5, dirt_probability=0.35, obstacle_probability=0.10, seed=404)
goal_result = run_simulation(GoalBasedVacuumAgent(), goal_env, max_steps=100)
goal_result

### Checkpoint 4 - Teaching Notes

The goal-based agent tries to reach the nearest dirty square, then clean it. BFS is appropriate because each move has the same cost and the grid can be represented as a graph of neighboring cells. This agent uses the full environment status map, so it has more information than the model-based agent. That means the performance comparison is useful pedagogically, but not completely fair if we are comparing agents with equal percept access.


## 10. Controlled Experiments

To compare agents fairly, each agent should face the same world in each trial. The function below creates one world, copies it, and then reloads that same world for each agent.

Complete or inspect the code, then run the experiment.

In [ ]:
def evaluate_agents(
    agent_factories,
    trials: int = 30,
    max_steps: int = 100,
    width: int = 5,
    height: int = 5,
    dirt_probability: float = 0.35,
    obstacle_probability: float = 0.10,
) -> List[dict]:
    rows = []

    for trial in range(trials):
        base_env = VacuumEnvironment(
            width=width,
            height=height,
            dirt_probability=dirt_probability,
            obstacle_probability=obstacle_probability,
            seed=1000 + trial,
        )
        status, obstacles = base_env.copy_world()

        for make_agent in agent_factories:
            env = VacuumEnvironment(width=width, height=height, dirt_probability=0.0, obstacle_probability=0.0)
            env.load_world(status, obstacles)
            agent = make_agent()
            result = run_simulation(agent, env, max_steps=max_steps)
            result["trial"] = trial
            rows.append(result)

    return rows


agent_factories = [
    lambda: RandomVacuumAgent(),
    lambda: ReflexVacuumAgent(),
    lambda: ModelBasedVacuumAgent(),
    lambda: GoalBasedVacuumAgent(),
]

results = evaluate_agents(agent_factories, trials=30, max_steps=100)
results[:4]

In [ ]:
def summarize_results(results: List[dict]) -> Dict[str, dict]:
    summary = {}
    agent_names = sorted({row["agent"] for row in results})

    for agent_name in agent_names:
        rows = [row for row in results if row["agent"] == agent_name]
        summary[agent_name] = {
            "avg_score": sum(row["score"] for row in rows) / len(rows),
            "avg_steps": sum(row["steps"] for row in rows) / len(rows),
            "avg_cleaned": sum(row["cleaned"] for row in rows) / len(rows),
            "avg_dirty_left": sum(row["dirty_left"] for row in rows) / len(rows),
        }

    return summary


summary = summarize_results(results)
summary

## 11. Plot Your Results

Create at least two plots:

1. Average score by agent.
2. Average dirty squares left by agent.

You may add more plots if they help your analysis.

In [ ]:
def plot_summary_metric(summary: Dict[str, dict], metric: str, title: str, ylabel: str) -> None:
    names = list(summary.keys())
    values = [summary[name][metric] for name in names]

    plt.figure(figsize=(9, 4))
    plt.bar(names, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


plot_summary_metric(summary, "avg_score", "Average Score by Agent", "Average score")
plot_summary_metric(summary, "avg_dirty_left", "Average Dirt Left by Agent", "Average dirty squares left")

### Checkpoint 5 - Teaching Notes

The goal-based agent should usually leave the least dirt behind and often has the strongest score because it plans directly toward dirty squares. The reflex and model-based agents may do well in open worlds, but obstacles and partial observability can make them inefficient. The highest score may not always mean the fewest steps because an agent can take more steps while cleaning more squares. The performance measure matters: if movement were free, wandering agents would be penalized less.


## 12. Extension Challenge

Choose one extension if you finish early.

### Option A: Backtracking Model-Based Agent

Improve the model-based agent by storing a stack of previous positions and backtracking when no unvisited neighbor is available.

### Option B: Random New Dirt

Modify the environment so that after each action, a clean square has a small probability of becoming dirty.

### Option C: New Performance Measure

Change the score rule. For example, make movement free, make bumps more costly, or add a larger bonus for finishing.

### Option D: Larger World

Run experiments on a 10x10 world. Explain what changes.

In [ ]:
# Extension workspace.
# Put your optional extension code here.

## 13. Final Reflection - Teaching Notes

A strong student reflection should explain that a rational agent chooses actions expected to maximize the performance measure, given its percept sequence and knowledge of the environment. The highest-scoring agent in one run is not automatically the most rational agent overall because random world generation, partial observability, and unfair information access can affect outcomes. Strong answers should connect results to AIMA vocabulary such as percepts, performance measure, environment properties, and rational action.

## Submission Checklist for Student Version

Before students submit, they should confirm that:

- The notebook runs from top to bottom without errors.
- All implementation sections are completed or clearly attempted.
- All five checkpoints are answered.
- The experiment summary is visible.
- The plots are visible.
- The final reflection is complete.
